# DSFB SRD Figure Regeneration

This notebook runs `dsfb-srd-generate` from scratch inside Colab, then reads a selected timestamped directory from `output-dsfb-srd/<timestamp>/` and regenerates the phase-transition figures for the deterministic Structural Regime Dynamics demonstrator.

The setup cell always executes the Rust crate first. Leave `RUN_NAME = None` to use the fresh run it just generated, or set `RUN_NAME` to inspect a specific timestamped run after the new run completes.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": ":",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

REPO_ROOT = Path("/content/dsfb")
RUN_NAME = None  # Optional: "20260312-214530"
AUTO_CLONE_REPO = True
DSFB_REPO_URL = "https://github.com/infinityabundance/dsfb.git"

def run_checked(command, cwd=None, env=None):
    print("+", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def unique_paths(paths):
    ordered = []
    seen = set()
    for candidate in paths:
        candidate = Path(candidate).expanduser()
        if candidate not in seen:
            seen.add(candidate)
            ordered.append(candidate)
    return ordered

def resolve_repo_root():
    candidates = []
    if REPO_ROOT is not None:
        candidates.append(Path(REPO_ROOT).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path("/content/dsfb"))

    for candidate in unique_paths(candidates):
        if (candidate / "crates/dsfb-srd/Cargo.toml").exists():
            return candidate
    return None

def ensure_repo_root():
    repo_root = resolve_repo_root()
    if repo_root is not None:
        return repo_root

    if not AUTO_CLONE_REPO:
        raise FileNotFoundError("DSFB repository not found and AUTO_CLONE_REPO is disabled")

    target = Path(REPO_ROOT).expanduser() if REPO_ROOT is not None else Path("/content/dsfb")
    if target.exists() and not (target / "crates/dsfb-srd/Cargo.toml").exists():
        raise FileExistsError(
            f"Cannot clone DSFB into {target}: path exists but is not the DSFB repository"
        )

    if not target.exists():
        run_checked(["git", "clone", "--depth", "1", DSFB_REPO_URL, str(target)])

    return target

def cargo_env():
    env = os.environ.copy()
    cargo_bin = str(Path.home() / ".cargo" / "bin")
    env["PATH"] = cargo_bin + os.pathsep + env.get("PATH", "")
    return env

def ensure_cargo():
    env = cargo_env()
    if shutil.which("cargo", path=env["PATH"]):
        return env

    run_checked(["sh", "-c", "curl https://sh.rustup.rs -sSf | sh -s -- -y"], env=env)

    if not shutil.which("cargo", path=env["PATH"]):
        raise RuntimeError("Rust installation completed but cargo is still unavailable")

    return env

def list_run_dirs(output_root):
    if not output_root.exists():
        return []
    return sorted(path for path in output_root.iterdir() if path.is_dir())

def run_generator_from_scratch(repo_root):
    env = ensure_cargo()
    output_root = repo_root / "output-dsfb-srd"
    before = {path.name for path in list_run_dirs(output_root)}

    run_checked(
        [
            "cargo",
            "run",
            "--manifest-path",
            "crates/dsfb-srd/Cargo.toml",
            "--release",
            "--bin",
            "dsfb-srd-generate",
        ],
        cwd=repo_root,
        env=env,
    )

    after = list_run_dirs(output_root)
    if not after:
        raise FileNotFoundError(
            f"No timestamped runs found under {output_root} after running dsfb-srd-generate"
        )

    fresh_runs = [path for path in after if path.name not in before]
    if not fresh_runs:
        raise RuntimeError(
            "dsfb-srd-generate completed but no new timestamped output directory was created"
        )

    return output_root, fresh_runs[-1]

def select_run_dir(output_root, fresh_run_dir):
    if RUN_NAME is None:
        return fresh_run_dir

    run_dir = output_root / RUN_NAME
    if not run_dir.exists():
        raise FileNotFoundError(f"Requested run directory not found: {run_dir}")
    return run_dir

REPO_ROOT = ensure_repo_root()
OUTPUT_ROOT, FRESH_RUN_DIR = run_generator_from_scratch(REPO_ROOT)
RUN_DIR = select_run_dir(OUTPUT_ROOT, FRESH_RUN_DIR)

print(f"Using repository: {REPO_ROOT}")
print(f"Using output root: {OUTPUT_ROOT}")
print(f"Fresh run directory: {FRESH_RUN_DIR}")
print(f"Selected run directory: {RUN_DIR}")

In [ ]:
manifest = pd.read_csv(RUN_DIR / "run_manifest.csv")
events = pd.read_csv(RUN_DIR / "events.csv")
threshold_sweep = pd.read_csv(RUN_DIR / "threshold_sweep.csv")
transition_sharpness = pd.read_csv(RUN_DIR / "transition_sharpness.csv")
time_local = pd.read_csv(RUN_DIR / "time_local_metrics.csv")

snapshot_020 = pd.read_csv(RUN_DIR / "graph_snapshot_tau_020.csv")
snapshot_030 = pd.read_csv(RUN_DIR / "graph_snapshot_tau_030.csv")
snapshot_040 = pd.read_csv(RUN_DIR / "graph_snapshot_tau_040.csv")

manifest_row = manifest.iloc[0]
shock_start = int(manifest_row["shock_start"])
shock_end = int(manifest_row["shock_end"])

time_local = time_local.copy()
time_local["window_midpoint"] = 0.5 * (time_local["window_start"] + time_local["window_end"])

def grouped_color_items(frame, group_key, sort_key):
    groups = sorted(frame.groupby(group_key), key=lambda item: item[0])
    cmap = plt.get_cmap("viridis")
    denominator = max(len(groups) - 1, 1)
    for index, (group_value, subset) in enumerate(groups):
        yield cmap(index / denominator), group_value, subset.sort_values(sort_key)

def plot_snapshot(frame, requested_tau, title_prefix):
    fig, ax = plt.subplots(figsize=(10, 6))
    if frame.empty:
        ax.text(0.5, 0.5, "No active edges at this threshold", ha="center", va="center", transform=ax.transAxes)
        actual_tau = requested_tau
    else:
        actual_tau = float(frame["tau_threshold"].iloc[0])
        ax.scatter(frame["src"], frame["dst"], s=5, alpha=0.55, color="#1f77b4", linewidths=0)
        ax.set_xlim(-5, max(frame["src"].max(), frame["dst"].max()) + 5)
        ax.set_ylim(-5, max(frame["src"].max(), frame["dst"].max()) + 5)

    ax.set_title(f"{title_prefix} (requested tau={requested_tau:.2f}, actual tau={actual_tau:.4f})")
    ax.set_xlabel("source event index")
    ax.set_ylabel("destination event index")
    fig.tight_layout()
    plt.show()

manifest

In [ ]:
fig, ax = plt.subplots()
for color, n_events, subset in grouped_color_items(threshold_sweep, "n_events", "tau_threshold"):
    ax.plot(
        subset["tau_threshold"],
        subset["reachable_fraction"],
        linewidth=2,
        color=color,
        label=f"N={n_events}",
    )

ax.set_title("Figure 1: Connectivity Collapse rho(tau)")
ax.set_xlabel("Trust threshold tau")
ax.set_ylabel("Reachable fraction rho(tau)")
ax.legend(title="event history")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for color, n_events, subset in grouped_color_items(transition_sharpness, "n_events", "tau_midpoint"):
    ax.plot(
        subset["tau_midpoint"],
        subset["abs_drho_dtau"],
        linewidth=2,
        color=color,
        label=f"N={n_events}",
    )

ax.set_title("Figure 2: Transition Sharpness |drho/dtau|")
ax.set_xlabel("Threshold midpoint")
ax.set_ylabel("Absolute derivative")
ax.legend(title="event history")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for color, n_events, subset in grouped_color_items(threshold_sweep, "n_events", "tau_threshold"):
    ax.plot(
        subset["tau_threshold"],
        subset["largest_component_fraction"],
        linewidth=2,
        color=color,
        label=f"N={n_events}",
    )

ax.set_title("Figure 3: Global Graph Coherence G(tau)")
ax.set_xlabel("Trust threshold tau")
ax.set_ylabel("Largest component fraction G(tau)")
ax.legend(title="event history")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for color, n_events, subset in grouped_color_items(threshold_sweep, "n_events", "tau_threshold"):
    ax.plot(
        subset["tau_threshold"],
        subset["component_entropy"],
        linewidth=2,
        color=color,
        label=f"N={n_events}",
    )

ax.set_title("Figure 4: Shannon Entropy of Weak Component Sizes")
ax.set_xlabel("Trust threshold tau")
ax.set_ylabel("Component entropy H(tau)")
ax.legend(title="event history")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
threshold_groups = sorted(time_local.groupby("tau_threshold"), key=lambda item: item[0])
color_cycle = plt.get_cmap("plasma")
denominator = max(len(threshold_groups) - 1, 1)

for index, (tau_threshold, subset) in enumerate(threshold_groups):
    subset = subset.sort_values("window_midpoint")
    ax.plot(
        subset["window_midpoint"],
        subset["reachable_fraction"],
        linewidth=2,
        color=color_cycle(index / denominator),
        label=f"tau={tau_threshold:.3f}",
    )

ax.axvspan(shock_start, shock_end, color="tomato", alpha=0.16, label="shock interval")
ax.set_title("Figure 5: Time-Local Connectivity During Structural Regimes")
ax.set_xlabel("Event index")
ax.set_ylabel("Window reachable fraction")
ax.legend(title="threshold")
fig.tight_layout()
plt.show()

In [ ]:
plot_snapshot(snapshot_020, 0.20, "Figure 6a: Network Snapshot in the Connected Regime")

In [ ]:
plot_snapshot(snapshot_030, 0.30, "Figure 6b: Network Snapshot in the Transition Regime")

In [ ]:
plot_snapshot(snapshot_040, 0.40, "Figure 6c: Network Snapshot in the Fragmented Regime")